# PayGuard — Model 5: One-Class SVM

**Unsupervised**, boundary/density-based anomaly detection — a completely different mechanism from Isolation Forest's tree-splitting approach. One-Class SVM learns a decision boundary that encloses the "normal" region of the feature space (using an RBF kernel to allow a non-linear boundary); anything falling outside that boundary is flagged as an anomaly.

**Important practical note:** RBF-kernel SVMs scale roughly O(n²) to O(n³) with training set size. Fitting on the full 64,000-row training set would be impractically slow. We train on a **random subsample** of the training set instead (a standard, documented practice for kernel SVMs on large data) and evaluate on the full, untouched test set as usual — only training is subsampled, not evaluation.

In [1]:
import joblib
import os
import numpy as np
from sklearn.svm import OneClassSVM
from evaluation import evaluate_model, print_comparison_table

np.random.seed(42)

bundle = joblib.load("../data/processed/train_test_variants.pkl")

X_train_scaled = bundle["X_train_scaled"]
y_train = bundle["y_train"]  # only for reference / computing the true fraud rate
X_test_scaled = bundle["X_test_scaled"]
y_test = bundle["y_test"]

true_fraud_rate = y_train.mean()

# Random subsample for training only (purely random — not using labels to pick rows,
# so this stays a genuinely unsupervised setup)
SUBSAMPLE_SIZE = 6000
subsample_idx = np.random.choice(X_train_scaled.index, size=SUBSAMPLE_SIZE, replace=False)
X_train_sub = X_train_scaled.loc[subsample_idx]

print(f"True training fraud rate (for reference only): {true_fraud_rate:.4f}")
print(f"Training on a subsample: {X_train_sub.shape[0]:,} rows (full train set was {X_train_scaled.shape[0]:,})")
print(f"Evaluating on full test set: {X_test_scaled.shape[0]:,} rows")

True training fraud rate (for reference only): 0.0028
Training on a subsample: 6,000 rows (full train set was 64,000)
Evaluating on full test set: 16,000 rows


## Helper function

Same conversion logic as Isolation Forest: `predict()` returns `-1` for anomaly / `1` for normal, and `decision_function()` is higher for "more normal" — both get flipped/converted to match our standard format.

`nu` is One-Class SVM's equivalent of Isolation Forest's `contamination` — an upper bound on the expected fraction of training outliers.

In [2]:
def run_one_class_svm(nu, label):
    model = OneClassSVM(kernel="rbf", gamma="scale", nu=nu)
    model.fit(X_train_sub)  # NOTE: no y_train passed, and trained on the subsample only

    raw_pred = model.predict(X_test_scaled)             # -1 = anomaly, 1 = normal
    y_pred = np.where(raw_pred == -1, 1, 0)              # convert to 1 = fraud, 0 = not fraud
    y_score = -model.decision_function(X_test_scaled)    # flip sign: higher = more anomalous

    result = evaluate_model(y_test, y_pred, "One-Class SVM", label, y_score)
    return model, result

## Variant 1: `nu` calibrated to the true training fraud rate

In [3]:
svm_calibrated, result_calibrated = run_one_class_svm(true_fraud_rate, "Calibrated to True Rate")


  One-Class SVM (Calibrated to True Rate)
Precision: 0.0488   Recall: 0.1778   F1: 0.0766

Confusion Matrix:
                 Predicted Not Fraud   Predicted Fraud
Actual Not Fraud           15799                 156
Actual Fraud                  37                   8

ROC-AUC: 0.6561

Full classification report:
              precision    recall  f1-score   support

   Not Fraud       1.00      0.99      0.99     15955
       Fraud       0.05      0.18      0.08        45

    accuracy                           0.99     16000
   macro avg       0.52      0.58      0.54     16000
weighted avg       0.99      0.99      0.99     16000



## Variant 2: Moderate assumption (`nu=0.05`)

One-Class SVM has no `"auto"` option like Isolation Forest, so we use a commonly-used moderate default instead — representing "we don't know the true rate, so assume a modest 5% outlier fraction."

In [4]:
svm_moderate, result_moderate = run_one_class_svm(0.05, "Moderate Assumption (nu=0.05)")


  One-Class SVM (Moderate Assumption (nu=0.05))
Precision: 0.0260   Recall: 0.4222   F1: 0.0489

Confusion Matrix:
                 Predicted Not Fraud   Predicted Fraud
Actual Not Fraud           15242                 713
Actual Fraud                  26                  19

ROC-AUC: 0.7298

Full classification report:
              precision    recall  f1-score   support

   Not Fraud       1.00      0.96      0.98     15955
       Fraud       0.03      0.42      0.05        45

    accuracy                           0.95     16000
   macro avg       0.51      0.69      0.51     16000
weighted avg       1.00      0.95      0.97     16000



## Variant 3: Wider net (2x the true rate)

In [5]:
svm_wide, result_wide = run_one_class_svm(min(true_fraud_rate * 2, 0.5), "Wider Net (2x Rate)")


  One-Class SVM (Wider Net (2x Rate))
Precision: 0.0491   Recall: 0.1778   F1: 0.0769

Confusion Matrix:
                 Predicted Not Fraud   Predicted Fraud
Actual Not Fraud           15800                 155
Actual Fraud                  37                   8

ROC-AUC: 0.6572

Full classification report:
              precision    recall  f1-score   support

   Not Fraud       1.00      0.99      0.99     15955
       Fraud       0.05      0.18      0.08        45

    accuracy                           0.99     16000
   macro avg       0.52      0.58      0.54     16000
weighted avg       0.99      0.99      0.99     16000



## Compare the three `nu` settings

In [6]:
svm_results = [result_calibrated, result_moderate, result_wide]
svm_comparison_df = print_comparison_table(svm_results)


  MODEL COMPARISON (sorted by F1)
        model                       variant  precision  recall     f1  roc_auc  true_positives  false_positives  true_negatives  false_negatives
One-Class SVM           Wider Net (2x Rate)     0.0491  0.1778 0.0769   0.6572               8              155           15800               37
One-Class SVM       Calibrated to True Rate     0.0488  0.1778 0.0766   0.6561               8              156           15799               37
One-Class SVM Moderate Assumption (nu=0.05)     0.0260  0.4222 0.0489   0.7298              19              713           15242               26


## Save the best variant's model + results

In [7]:
os.makedirs("../models", exist_ok=True)

best_variant = svm_comparison_df.iloc[0]
print(f"Best variant for One-Class SVM: {best_variant['variant']} (F1={best_variant['f1']}, recall={best_variant['recall']})")

variant_to_model = {
    "Calibrated to True Rate": svm_calibrated,
    "Moderate Assumption (nu=0.05)": svm_moderate,
    "Wider Net (2x Rate)": svm_wide,
}
best_model = variant_to_model[best_variant["variant"]]

joblib.dump(best_model, "../models/one_class_svm.pkl")
svm_comparison_df.to_csv("../models/one_class_svm_results.csv", index=False)
print("Saved model to ../models/one_class_svm.pkl")
print("Saved results to ../models/one_class_svm_results.csv")

Best variant for One-Class SVM: Wider Net (2x Rate) (F1=0.0769, recall=0.1778)
Saved model to ../models/one_class_svm.pkl
Saved results to ../models/one_class_svm_results.csv


## All five models are now trained. Next: `05_final_model_comparison.ipynb`

Loads every model's saved results CSV and builds the single grand comparison table across all five — the centerpiece table for your report and viva.